In [ ]:
# ============================================================
# Cell 1: 初始化环境（多GPU版）
# ============================================================
import os
import gc
import json
import sys
import importlib
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GATv2Conv
from torch_geometric.utils import from_scipy_sparse_matrix

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.neighbors import kneighbors_graph, BallTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import StandardScaler as SKScaler
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from datetime import datetime
from openpyxl import load_workbook

import warnings

warnings.filterwarnings("ignore")

# ── 随机种子 ──────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)

# ── 检测所有可用GPU ────────────────────────────────────────
NUM_GPUS = torch.cuda.device_count()
print(f"检测到 {NUM_GPUS} 张GPU")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    free_mem = torch.cuda.mem_get_info(i)[0] / 1024**3
    print(
        f"  GPU {i}: {props.name} | 总显存: {props.total_memory/1024**3:.1f}GB | 可用: {free_mem:.1f}GB"
    )

# ── 设备列表（优先GPU，无GPU退回CPU）──────────────────────
if NUM_GPUS > 0:
    DEVICES = [torch.device(f"cuda:{i}") for i in range(NUM_GPUS)]
    MAX_PARALLEL = NUM_GPUS
else:
    DEVICES = [torch.device("cpu")]
    MAX_PARALLEL = 1
    print("警告：未检测到GPU，将使用CPU运行")

device_GPU = DEVICES[0]
device_CPU = torch.device("cpu")

print(f"\n并行任务数: {MAX_PARALLEL}")
print(f"主设备: {device_GPU}")

# ── multiprocessing启动方式（必须在最顶层设置）────────────
mp.set_start_method("spawn", force=True)

# ── PyTorch环境变量优化 ───────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
cpu_cores = os.cpu_count()
torch.set_num_threads(max(1, cpu_cores // MAX_PARALLEL))

In [ ]:
# ============================================================
# Cell 2: 目录管理
# ★ 去掉 MLP，加入 TopACT
# ============================================================
print("\n" + "=" * 60)
print("0. 初始化实验目录结构")
print("=" * 60)

BASE_DIR = "./"
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")
CURVES_DIR = os.path.join(FIGURES_DIR, "training_curves")
CONFUSION_DIR = os.path.join(FIGURES_DIR, "confusion_matrices")

for d in [BASE_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR, CURVES_DIR, CONFUSION_DIR]:
    os.makedirs(d, exist_ok=True)

# ★ 去掉 MLP，加入 TopACT
MODEL_DIRS = {
    "GCN": os.path.join(MODELS_DIR, "GCN"),
    "GraphSAGE": os.path.join(MODELS_DIR, "GraphSAGE"),
    "GAT": os.path.join(MODELS_DIR, "GAT"),
    "GATv2": os.path.join(MODELS_DIR, "GATv2"),
    "TopACT": os.path.join(MODELS_DIR, "TopACT"),
}
for d in MODEL_DIRS.values():
    os.makedirs(d, exist_ok=True)

# ★ 去掉 MLP，加入 TopACT
EXCEL_PATHS = {
    "Xenium": "./results/exports/xenium_for_gnn.csv",
    "GCN": os.path.join(MODEL_DIRS["GCN"], "GCN_results.xlsx"),
    "GraphSAGE": os.path.join(MODEL_DIRS["GraphSAGE"], "GraphSAGE_results.xlsx"),
    "GAT": os.path.join(MODEL_DIRS["GAT"], "GAT_results.xlsx"),
    "GATv2": os.path.join(MODEL_DIRS["GATv2"], "GATv2_results.xlsx"),
    "TopACT": os.path.join(MODEL_DIRS["TopACT"], "TopACT_results.xlsx"),
}

FIGURE_PATHS = {
    "validation_comparison": os.path.join(
        FIGURES_DIR, "model_validation_comparison.png"
    ),
    "test_comparison": os.path.join(FIGURES_DIR, "model_test_accuracy_comparison.png"),
    "f1_comparison": os.path.join(FIGURES_DIR, "model_f1_comparison.png"),
    "parameter_analysis": os.path.join(FIGURES_DIR, "parameter_analysis.png"),
}

print("✓ 目录初始化完成")

In [ ]:
# ============================================================
# Cell 3: 辅助函数
# ============================================================
def save_model_and_results(model, config, model_name, training_history=None):
    model_filename = (
        f"{model_name}_h{config['hidden_dim']}_d{config['dropout']}"
        f"_l{config['num_layers']}"
    )
    if "heads" in config:
        model_filename += f"_heads{config['heads']}"
    if "use_skip" in config:
        model_filename += f"_skip{config['use_skip']}"
    model_filename += ".pt"

    os.makedirs(MODEL_DIRS[model_name], exist_ok=True)
    model_path = os.path.join(MODEL_DIRS[model_name], model_filename)

    save_dict = {
        "model_state_dict": model.state_dict(),
        "config": config,
        "best_val_acc": config["best_val_acc"],
        "test_acc": config["test_acc"],
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    if training_history:
        save_dict["training_history"] = training_history
    torch.save(save_dict, model_path)

    result_row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "hidden_dim": config["hidden_dim"],
        "dropout": config["dropout"],
        "num_layers": config["num_layers"],
        "best_val_acc": config["best_val_acc"],
        "test_acc": config["test_acc"],
        "test_f1_macro": config["test_f1_macro"],
        "test_f1_weighted": config["test_f1_weighted"],
        "test_precision_macro": config.get("test_precision_macro", 0),
        "test_recall_macro": config.get("test_recall_macro", 0),
        "num_params": config.get("num_params", 0),
        "model_file": model_filename,
    }
    if "heads" in config:
        result_row["heads"] = config["heads"]
    if "use_skip" in config:
        result_row["use_skip"] = config["use_skip"]

    excel_path = EXCEL_PATHS[model_name]
    if os.path.exists(excel_path):
        try:
            existing_df = pd.read_excel(excel_path)
            results_df = pd.concat(
                [existing_df, pd.DataFrame([result_row])], ignore_index=True
            )
        except Exception:
            results_df = pd.DataFrame([result_row])
    else:
        results_df = pd.DataFrame([result_row])

    results_df = results_df.sort_values("best_val_acc", ascending=False)
    results_df.to_excel(excel_path, index=False)
    print(f"  ✓ 模型已保存: {model_path}")
    return model_path


def save_detailed_results(
    model_name, config, training_history, test_true, test_pred, class_names
):
    precision, recall, f1, support = precision_recall_fscore_support(
        test_true, test_pred, average=None
    )
    detailed_results = {
        "model_name": model_name,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "config": {
            k: v
            for k, v in config.items()
            if k
            not in ["test_true", "test_pred", "test_pred_labels", "test_true_labels"]
        },
        "best_val_acc": float(config["best_val_acc"]),
        "test_metrics": {
            "accuracy": float(config["test_acc"]),
            "f1_macro": float(config["test_f1_macro"]),
            "f1_weighted": float(config["test_f1_weighted"]),
            "precision_macro": float(config.get("test_precision_macro", 0)),
            "recall_macro": float(config.get("test_recall_macro", 0)),
        },
        "per_class_metrics": {},
    }
    for i in range(len(class_names)):
        detailed_results["per_class_metrics"][f"class_{i}"] = {
            "class_name": str(class_names[i]),
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
    if training_history:
        detailed_results["training_history_summary"] = {
            "final_train_loss": (
                float(training_history["train_losses"][-1])
                if training_history["train_losses"]
                else None
            ),
            "final_val_loss": (
                float(training_history["val_losses"][-1])
                if training_history["val_losses"]
                else None
            ),
            "best_val_acc": float(config["best_val_acc"]),
            "best_val_loss": (
                float(min(training_history["val_losses"]))
                if training_history["val_losses"]
                else None
            ),
            "epochs_trained": len(training_history["train_losses"]),
        }
    param_str = f"h{config['hidden_dim']}_d{config['dropout']}_l{config['num_layers']}"
    if "heads" in config:
        param_str += f"_heads{config['heads']}"
    filename = f"{model_name}_{param_str}.json"
    save_path = os.path.join(MODEL_DIRS[model_name], filename)
    with open(save_path, "w") as f:
        json.dump(detailed_results, f, indent=2)
    return save_path

In [ ]:
# ============================================================
# Cell 4: 数据加载、特征提取、建图、数据集划分
# ============================================================

# ── 1. 加载数据 ───────────────────────────────────────────
print("\n" + "=" * 60)
print("1. 加载数据")
print("=" * 60)
df = pd.read_csv(EXCEL_PATHS["Xenium"])
df = df.dropna()
print(f"数据形状（去除缺失值后）: {df.shape}")

# ── 2. 提取特征和标签 ──────────────────────────────────────
print("\n" + "=" * 60)
print("2. 提取特征和标签")
print("=" * 60)
coords = df[["x_centroid", "y_centroid"]].values
pca_cols = [col for col in df.columns if col.startswith("PC_")]
features = df[pca_cols].values
scaler = StandardScaler()
features = scaler.fit_transform(features)
labels_raw = df["predicted_cell_type"].values
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels_raw)
num_classes = len(label_encoder.classes_)
class_names = label_encoder.classes_.tolist()
print(f"PCA特征维度: {features.shape} | 类别数: {num_classes}")

# ── 3. 建图 ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("3. 构建空间邻接图")
print("=" * 60)


def build_spatial_graph(coords, k=10):
    adj_matrix = kneighbors_graph(
        coords, n_neighbors=k, mode="connectivity", include_self=False
    )
    edge_index, _ = from_scipy_sparse_matrix(adj_matrix)
    return edge_index


DEFAULT_K = 5
edge_index = build_spatial_graph(coords, k=DEFAULT_K)
print(f"使用 k={DEFAULT_K} | 边数量: {edge_index.shape[1]}")

# ── 4. 创建PyG数据对象 ────────────────────────────────────
x = torch.FloatTensor(features)
y = torch.LongTensor(labels)
edge_index = edge_index.to(torch.long)
data = Data(x=x, edge_index=edge_index, y=y)
print(
    f"节点数: {data.num_nodes} | 特征维度: {data.num_features} | 边数: {data.num_edges}"
)

# ── 5. 划分数据集 ─────────────────────────────────────────
indices = np.arange(data.num_nodes)
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.2, stratify=labels, random_state=42
)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.25, stratify=labels[train_val_idx], random_state=42
)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
val_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask
print(
    f"训练集: {train_mask.sum().item()} | 验证集: {val_mask.sum().item()} | 测试集: {test_mask.sum().item()}"
)

In [ ]:
# ============================================================
# Cell 5: GNN 模型定义（去掉 MLPWithDropout）
# ============================================================
print("\n" + "=" * 60)
print("6. 定义图神经网络模型")
print("=" * 60)


class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropout = dropout
        self.convs.append(GCNConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(GCNConv(hidden_dim, out_dim))
        self.skip_proj = nn.Linear(in_dim, hidden_dim) if num_layers >= 2 else None

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        identity = x
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if i == 0 and self.skip_proj is not None:
                x = x + self.skip_proj(identity)
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


class GraphSAGEWithNorm(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropout = dropout
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(SAGEConv(hidden_dim, out_dim))

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


class GAT(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden_dim,
        out_dim,
        heads=8,
        dropout=0.3,
        num_layers=2,
        use_skip=True,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.use_skip = use_skip
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(
            GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True)
        )
        self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        for _ in range(num_layers - 2):
            self.convs.append(
                GATConv(
                    hidden_dim * heads,
                    hidden_dim,
                    heads=heads,
                    dropout=dropout,
                    concat=True,
                )
            )
            self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        self.convs.append(
            GATConv(hidden_dim * heads, out_dim, heads=1, dropout=dropout, concat=False)
        )
        if use_skip and num_layers >= 2:
            self.skip_proj = nn.Linear(in_dim, hidden_dim * heads)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        identity = (
            self.skip_proj(x) if (self.use_skip and self.num_layers >= 2) else None
        )
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            if identity is not None and i == 0:
                x = x + identity
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


class GATv2Model(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, dropout=0.3, num_layers=2):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(
            GATv2Conv(in_dim, hidden_dim, heads=heads, dropout=dropout, concat=True)
        )
        self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        for _ in range(num_layers - 2):
            self.convs.append(
                GATv2Conv(
                    hidden_dim * heads,
                    hidden_dim,
                    heads=heads,
                    dropout=dropout,
                    concat=True,
                )
            )
            self.bns.append(nn.BatchNorm1d(hidden_dim * heads))
        self.convs.append(
            GATv2Conv(
                hidden_dim * heads, out_dim, heads=1, dropout=dropout, concat=False
            )
        )

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return F.log_softmax(self.convs[-1](x, edge_index), dim=1)


print("所有GNN模型定义完成")

In [ ]:
# ============================================================
# Cell 6: TopACT 核心类实现
# ============================================================
class TopACT:
    """
    TopACT: Topological Automatic Cell Type identification
    复现自 Benjamin et al., Nature 2024

    工作流程：
    1. 用 scRNA-seq 参考数据（或已标注空间点）训练一个局部分类器（SVM）
    2. 对每个空间点，在半径 r1 < r2 < ... < rk 下逐步聚合邻域表达向量
    3. 将聚合向量输入 SVM 得到各细胞类型的置信度向量
    4. 取首个超过阈值 θ 的预测结果作为该点的细胞类型

    参数说明：
        radii      : list[float]，多尺度半径列表（单位与坐标相同）
        theta      : float，置信度阈值（论文默认 0.9）
        svm_C      : SVM 正则化系数
        svm_kernel : SVM 核函数
        use_platt  : 是否用 Platt scaling 输出概率
    """

    def __init__(
        self,
        radii=None,
        theta=0.5,
        svm_C=1.0,
        svm_kernel="rbf",
        use_platt=True,
        n_jobs=-1,
    ):
        self.radii = radii
        self.theta = theta
        self.svm_C = svm_C
        self.svm_kernel = svm_kernel
        self.use_platt = use_platt
        self.n_jobs = n_jobs

        self.classifier_ = None
        self.classes_ = None
        self.ball_tree_ = None
        self.coords_ = None
        self.features_ = None

    def fit(self, X_train, y_train, coords_all, X_all):
        """
        训练 TopACT 的局部 SVM 分类器，并建立全局空间索引。

        参数：
            X_train   : (N_train, D) 训练集特征
            y_train   : (N_train,)   训练集标签（整数编码）
            coords_all: (N_all, 2)   所有空间点的 (x, y) 坐标
            X_all     : (N_all, D)   所有空间点的表达特征
        """
        print("  [TopACT] 训练局部 SVM 分类器...")
        self.classes_ = np.unique(y_train)

        self.classifier_ = Pipeline(
            [
                ("scaler", SKScaler()),
                (
                    "svm",
                    SVC(
                        C=self.svm_C,
                        kernel=self.svm_kernel,
                        probability=self.use_platt,
                        decision_function_shape="ovr",
                        random_state=42,
                        cache_size=2000,
                    ),
                ),
            ]
        )
        self.classifier_.fit(X_train, y_train)
        print(f"  [TopACT] SVM 训练完成，类别数: {len(self.classes_)}")

        print("  [TopACT] 建立空间索引 (BallTree)...")
        self.coords_ = np.asarray(coords_all, dtype=np.float64)
        self.features_ = np.asarray(X_all, dtype=np.float32)
        self.ball_tree_ = BallTree(self.coords_, metric="euclidean")

        if self.radii is None:
            self.radii = self._estimate_radii()
            print(f"  [TopACT] 自动估算多尺度半径: {[f'{r:.2f}' for r in self.radii]}")

        return self

    def _estimate_radii(self, n_scales=5):
        """根据数据的 k-NN 距离自动估算合理的多尺度半径。"""
        dists_1, _ = self.ball_tree_.query(self.coords_[:1000], k=2)
        dists_10, _ = self.ball_tree_.query(self.coords_[:1000], k=11)
        r_min = float(np.median(dists_1[:, 1]))
        r_max = float(np.median(dists_10[:, 10]))
        radii = list(np.linspace(r_min, r_max, n_scales))
        return radii

    def predict(self, coords_query=None, X_query=None, batch_size=5000, verbose=True):
        """
        对查询点执行 TopACT 多尺度推理。

        返回：
            pred_labels : (N_query,) int
            confidence  : (N_query,) float
            pred_scale  : (N_query,) int，做出预测时所用的半径索引
        """
        if coords_query is None:
            coords_query = self.coords_
            X_query = self.features_

        coords_query = np.asarray(coords_query, dtype=np.float64)
        X_query = np.asarray(X_query, dtype=np.float32)
        N = len(coords_query)

        pred_labels = np.full(N, -1, dtype=int)
        confidence = np.zeros(N, dtype=float)
        pred_scale = np.full(N, len(self.radii) - 1, dtype=int)

        if verbose:
            print(
                f"  [TopACT] 开始多尺度推理，点数={N}，半径={len(self.radii)} 个尺度..."
            )

        for batch_start in range(0, N, batch_size):
            batch_end = min(batch_start + batch_size, N)
            batch_coords = coords_query[batch_start:batch_end]
            batch_X_base = X_query[batch_start:batch_end]
            batch_size_actual = batch_end - batch_start
            decided = np.zeros(batch_size_actual, dtype=bool)

            for scale_idx, r in enumerate(self.radii):
                if decided.all():
                    break

                neighbor_indices = self.ball_tree_.query_radius(batch_coords, r=r)
                X_aggregated = np.zeros(
                    (batch_size_actual, self.features_.shape[1]), dtype=np.float32
                )
                for i, nbr_idx in enumerate(neighbor_indices):
                    if len(nbr_idx) > 0:
                        X_aggregated[i] = self.features_[nbr_idx].sum(axis=0)
                    else:
                        X_aggregated[i] = batch_X_base[i]

                proba = self.classifier_.predict_proba(X_aggregated)
                max_proba = proba.max(axis=1)
                max_class_idx = proba.argmax(axis=1)

                newly_decided = (~decided) & (max_proba >= self.theta)
                pred_labels[batch_start:batch_end][newly_decided] = self.classes_[
                    max_class_idx[newly_decided]
                ]
                confidence[batch_start:batch_end][newly_decided] = max_proba[
                    newly_decided
                ]
                pred_scale[batch_start:batch_end][newly_decided] = scale_idx
                decided[newly_decided] = True

            # 对仍未决策的点，强制用最大尺度的最高概率预测
            undecided = ~decided
            if undecided.any():
                r_max = self.radii[-1]
                neighbor_indices_max = self.ball_tree_.query_radius(
                    batch_coords, r=r_max
                )
                X_agg_max = np.zeros(
                    (batch_size_actual, self.features_.shape[1]), dtype=np.float32
                )
                for i, nbr_idx in enumerate(neighbor_indices_max):
                    if len(nbr_idx) > 0:
                        X_agg_max[i] = self.features_[nbr_idx].sum(axis=0)
                    else:
                        X_agg_max[i] = batch_X_base[i]
                proba_max = self.classifier_.predict_proba(X_agg_max)
                max_proba_f = proba_max.max(axis=1)
                max_class_f = proba_max.argmax(axis=1)

                for local_i in np.where(undecided)[0]:
                    global_i = batch_start + local_i
                    pred_labels[global_i] = self.classes_[max_class_f[local_i]]
                    confidence[global_i] = max_proba_f[local_i]
                    pred_scale[global_i] = len(self.radii) - 1

            if verbose and (batch_end % (batch_size * 5) == 0 or batch_end == N):
                decided_ratio = (pred_labels[batch_start:batch_end] != -1).mean()
                print(
                    f"  [TopACT] 已处理 {batch_end}/{N} 点 | 有效预测率: {decided_ratio:.1%}"
                )

        if verbose:
            print(f"  [TopACT] 推理完成 | 平均置信度: {confidence.mean():.4f}")

        return pred_labels, confidence, pred_scale

    def fit_predict(self, X_train, y_train, coords_all, X_all, **kwargs):
        """便捷函数：先训练再推理全体点"""
        self.fit(X_train, y_train, coords_all, X_all)
        return self.predict(**kwargs)

In [ ]:
# ============================================================
# Cell 7: TopACT 实验函数
# ============================================================
def run_topact_experiment(
    data,
    coords,
    features,
    labels,
    class_names,
    MODEL_DIRS,
    EXCEL_PATHS,
    theta=0.5,
    n_scales=5,
    svm_C=1.0,
):
    """
    在现有实验框架中运行 TopACT，接口与 GNN 实验块完全对齐。
    返回：config dict（与其他模型结果格式一致）
    """
    print("\n" + "=" * 60)
    print("TopACT 基线模型实验")
    print("=" * 60)

    train_mask_np = data.train_mask.numpy()
    val_mask_np = data.val_mask.numpy()
    test_mask_np = data.test_mask.numpy()

    X_train = features[train_mask_np]
    y_train = labels[train_mask_np]
    y_val = labels[val_mask_np]
    y_test = labels[test_mask_np]

    print(
        f"训练集: {X_train.shape[0]} | 验证集: {val_mask_np.sum()} | 测试集: {test_mask_np.sum()}"
    )
    print(f"特征维度: {features.shape[1]} | 类别数: {len(class_names)}")
    print(f"置信度阈值 θ={theta} | 多尺度数={n_scales} | SVM C={svm_C}")

    print("\n[1/3] 训练 TopACT（SVM局部分类器 + 空间索引）...")
    topact = TopACT(
        radii=None, theta=theta, svm_C=svm_C, svm_kernel="rbf", use_platt=True
    )
    topact.fit(X_train=X_train, y_train=y_train, coords_all=coords, X_all=features)

    print("\n[2/3] 全局多尺度推理（训练+验证+测试点）...")
    pred_all, conf_all, scale_all = topact.predict(
        coords_query=coords,
        X_query=features,
        batch_size=10000,
        verbose=True,
    )

    val_pred = pred_all[val_mask_np]
    val_acc = accuracy_score(y_val, val_pred)
    print(f"\n验证集准确率: {val_acc:.4f}")

    print("\n[3/3] 测试集评估...")
    test_pred = pred_all[test_mask_np]
    test_true = y_test

    test_acc = float(accuracy_score(test_true, test_pred))
    test_f1_macro = float(
        f1_score(test_true, test_pred, average="macro", zero_division=0)
    )
    test_f1_weighted = float(
        f1_score(test_true, test_pred, average="weighted", zero_division=0)
    )
    precision, recall, _, _ = precision_recall_fscore_support(
        test_true, test_pred, average="macro", zero_division=0
    )

    print(f"测试准确率:      {test_acc:.4f}")
    print(f"Macro F1:        {test_f1_macro:.4f}")
    print(f"Weighted F1:     {test_f1_weighted:.4f}")
    print(f"Macro Precision: {float(precision):.4f}")
    print(f"Macro Recall:    {float(recall):.4f}")

    test_conf = conf_all[test_mask_np]
    test_scale = scale_all[test_mask_np]
    print(f"\n置信度统计（测试集）:")
    print(f"  平均置信度: {test_conf.mean():.4f} | 最低: {test_conf.min():.4f}")
    decided_at_scale = {
        f"r{i}({topact.radii[i]:.2f})": int((test_scale == i).sum())
        for i in range(len(topact.radii))
    }
    print(f"  各尺度决策数: {decided_at_scale}")

    config = {
        "model_name": "TopACT",
        "hidden_dim": 0,
        "dropout": 0.0,
        "num_layers": n_scales,
        "best_val_acc": val_acc,
        "num_params": 0,
        "theta": theta,
        "svm_C": svm_C,
        "n_scales": n_scales,
        "radii": [float(r) for r in topact.radii],
        "test_acc": test_acc,
        "test_f1_macro": test_f1_macro,
        "test_f1_weighted": test_f1_weighted,
        "test_precision_macro": float(precision),
        "test_recall_macro": float(recall),
        "test_pred": test_pred.tolist(),
        "test_true": test_true.tolist(),
        "mean_confidence": float(test_conf.mean()),
    }

    os.makedirs(MODEL_DIRS["TopACT"], exist_ok=True)
    result_path = os.path.join(
        MODEL_DIRS["TopACT"], f"TopACT_theta{theta}_C{svm_C}_scales{n_scales}.json"
    )
    with open(result_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"\n✓ 结果已保存: {result_path}")

    result_row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "hidden_dim": 0,
        "dropout": 0.0,
        "num_layers": n_scales,
        "best_val_acc": val_acc,
        "test_acc": test_acc,
        "test_f1_macro": test_f1_macro,
        "test_f1_weighted": test_f1_weighted,
        "test_precision_macro": float(precision),
        "test_recall_macro": float(recall),
        "num_params": 0,
        "theta": theta,
        "svm_C": svm_C,
        "model_file": os.path.basename(result_path),
    }
    excel_path = EXCEL_PATHS.get(
        "TopACT", os.path.join(MODEL_DIRS["TopACT"], "TopACT_results.xlsx")
    )
    if os.path.exists(excel_path):
        try:
            existing_df = pd.read_excel(excel_path)
            results_df = pd.concat(
                [existing_df, pd.DataFrame([result_row])], ignore_index=True
            )
        except Exception:
            results_df = pd.DataFrame([result_row])
    else:
        results_df = pd.DataFrame([result_row])
    results_df = results_df.sort_values("best_val_acc", ascending=False)
    results_df.to_excel(excel_path, index=False)
    print(f"✓ Excel 已保存: {excel_path}")

    return config, topact


def run_topact_grid_search(
    data, coords, features, labels, class_names, MODEL_DIRS, EXCEL_PATHS
):
    """对 TopACT 进行超参网格搜索，返回 list[dict]。"""
    print("\n" + "=" * 60)
    print("TopACT 参数网格搜索")
    print("=" * 60)

    param_grid = {
        "theta": [0.5, 0.7, 0.9],
        "svm_C": [0.1, 1.0, 10.0],
        "n_scales": [3, 5],
    }

    from itertools import product as iproduct

    combos = list(
        iproduct(param_grid["theta"], param_grid["svm_C"], param_grid["n_scales"])
    )
    print(f"总参数组合数: {len(combos)}")

    topact_results = []
    for idx, (theta, svm_C, n_scales) in enumerate(combos):
        print(
            f"\n[{idx+1}/{len(combos)}] theta={theta}, svm_C={svm_C}, n_scales={n_scales}"
        )
        try:
            config, _ = run_topact_experiment(
                data=data,
                coords=coords,
                features=features,
                labels=labels,
                class_names=class_names,
                MODEL_DIRS=MODEL_DIRS,
                EXCEL_PATHS=EXCEL_PATHS,
                theta=theta,
                n_scales=n_scales,
                svm_C=svm_C,
            )
            topact_results.append(config)
            print(
                f"  ✓ Val={config['best_val_acc']:.4f} | Test={config['test_acc']:.4f} | F1={config['test_f1_macro']:.4f}"
            )
        except Exception as e:
            print(f"  ✗ 失败: {e}")

    if topact_results:
        best = max(topact_results, key=lambda x: x["best_val_acc"])
        print(f"\nTopACT 最佳配置:")
        print(
            f"  theta={best['theta']}, svm_C={best['svm_C']}, n_scales={best['n_scales']}"
        )
        print(f"  Val Acc={best['best_val_acc']:.4f} | Test Acc={best['test_acc']:.4f}")
        print(f"  Macro F1={best['test_f1_macro']:.4f}")

    return topact_results

In [ ]:
# ============================================================
# Cell 8: 训练与评估函数（多GPU自适应版）
# ============================================================
print("\n" + "=" * 60)
print("7. 定义训练和评估函数（多GPU版）")
print("=" * 60)


def clear_memory(device=None):
    if device is not None and device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
    gc.collect()


def train_model(model, data, device, optimizer, epochs=500, patience=50, verbose=True):
    """单卡训练函数（device可以是任意GPU或CPU）"""
    data = data.to(device)
    model = model.to(device)

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    best_val_acc = 0
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    val_interval = 1

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast("cuda"):
                out = model(data)
                loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(data)
            loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            pred = out.argmax(dim=1)
            train_acc = (
                (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            )
        train_losses.append(loss.item())
        train_accs.append(train_acc)

        if (epoch + 1) % val_interval == 0:
            model.eval()
            with torch.no_grad():
                out_val = model(data)
                val_loss = F.nll_loss(out_val[data.val_mask], data.y[data.val_mask])
                pred_val = out_val.argmax(dim=1)
                val_acc = (
                    (pred_val[data.val_mask] == data.y[data.val_mask])
                    .float()
                    .mean()
                    .item()
                )
            val_losses.append(val_loss.item())
            val_accs.append(val_acc)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_val_loss = val_loss.item()
                best_model_state = {
                    k: v.cpu().clone() for k, v in model.state_dict().items()
                }
                patience_counter = 0
            else:
                patience_counter += 1

            if patience_counter >= patience:
                if verbose:
                    print(f"  早停于 epoch {epoch+1}")
                break

            if verbose and (epoch + 1) % 50 == 0:
                print(
                    f"  Epoch {epoch+1:4d} | Loss: {loss.item():.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f}"
                )

        if use_amp and (epoch + 1) % 100 == 0:
            clear_memory(device)

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    return model, train_losses, val_losses, train_accs, val_accs, best_val_acc


def evaluate_model(model, data, device):
    model.eval()
    data = data.to(device)
    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)
        test_pred = pred[data.test_mask].cpu().numpy()
        test_true = data.y[data.test_mask].cpu().numpy()

    test_acc = accuracy_score(test_true, test_pred)
    test_f1_macro = f1_score(test_true, test_pred, average="macro")
    test_f1_weighted = f1_score(test_true, test_pred, average="weighted")
    precision, recall, _, _ = precision_recall_fscore_support(
        test_true, test_pred, average="macro"
    )

    return {
        "test_acc": float(test_acc),
        "test_f1_macro": float(test_f1_macro),
        "test_f1_weighted": float(test_f1_weighted),
        "test_precision_macro": float(precision),
        "test_recall_macro": float(recall),
        "test_pred": test_pred.tolist(),
        "test_true": test_true.tolist(),
    }


# ── 多GPU并行核心函数 ─────────────────────────────────────────

# 确保能找到gnn_worker.py
if "." not in sys.path:
    sys.path.insert(0, ".")

# 重新加载（防止多次运行时用到旧版本）
if "gnn_worker" in sys.modules:
    importlib.reload(sys.modules["gnn_worker"])

from gnn_worker import worker_process  # spawn子进程能正确找到


def run_parallel_experiments(
    model_name, param_combinations_list, data, class_names_list, num_gpus=None
):
    """多GPU并行参数搜索（任务并行：每GPU同时跑一个参数组合）"""
    if num_gpus is None:
        num_gpus = torch.cuda.device_count()
    num_gpus = max(1, min(num_gpus, torch.cuda.device_count()))

    total = len(param_combinations_list)
    print(f"\n{'='*60}")
    print(f"开始 {model_name} 参数搜索")
    print(f"总任务数: {total} | 使用GPU数: {num_gpus} | 理论加速: ~{num_gpus}x")
    print(f"{'='*60}")

    data_dict = {
        "x": data.x.cpu(),
        "edge_index": data.edge_index.cpu(),
        "y": data.y.cpu(),
        "train_mask": data.train_mask.cpu(),
        "val_mask": data.val_mask.cpu(),
        "test_mask": data.test_mask.cpu(),
    }

    ctx = mp.get_context("spawn")
    task_queue = ctx.Queue()
    result_queue = ctx.Queue()

    for idx, params in enumerate(param_combinations_list):
        task_queue.put((idx, params))
    for _ in range(num_gpus):
        task_queue.put(None)

    processes = []
    for gpu_id in range(num_gpus):
        p = ctx.Process(
            target=worker_process,
            args=(
                gpu_id,
                task_queue,
                result_queue,
                data_dict,
                model_name,
                class_names_list,
            ),
            daemon=True,
        )
        p.start()
        processes.append(p)

    results_dict = {}
    completed = 0
    while completed < total:
        idx, result, error = result_queue.get()
        completed += 1
        if error:
            print(f"  ✗ 任务 #{idx} 失败:\n{error}")
        else:
            results_dict[idx] = result
            print(
                f"  [{completed}/{total}] ✓ {model_name} #{idx} "
                f"Val={result['best_val_acc']:.4f} Test={result['test_acc']:.4f}"
            )

    for p in processes:
        p.join()

    results = [results_dict[i] for i in range(total) if i in results_dict]
    if results:
        best = max(results, key=lambda x: x["best_val_acc"])
        print(f"\n{model_name} 最佳: {best}")
    return results


print("✓ run_parallel_experiments 已定义")

In [ ]:
# ============================================================
# Cell 9: 参数网格配置
# ============================================================
print("\n" + "=" * 60)
print("8. 模型参数实验配置")
print("=" * 60)

in_dim = data.num_features
out_dim = num_classes

param_grids = {
    "GCN": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GraphSAGE": {
        "hidden_dim": [128, 256],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3, 4],
    },
    "GAT": {
        "hidden_dim": [64, 128],
        "heads": [4, 8],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3],
        "use_skip": [True],
    },
    "GATv2": {
        "hidden_dim": [64, 128],
        "heads": [4, 8],
        "dropout": [0.2, 0.25, 0.3],
        "num_layers": [2, 3],
    },
}
print("参数网格配置完成")


def expand_param_grid(grid):
    """将param_grid展开为 list of dict，便于并行分发"""
    keys = list(grid.keys())
    values = list(grid.values())
    return [dict(zip(keys, combo)) for combo in product(*values)]

In [ ]:
# ============================================================
# Cell 10: TopACT 参数网格搜索
# ============================================================
topact_results = run_topact_grid_search(
    data=data,
    coords=coords,
    features=features,
    labels=labels,
    class_names=class_names,
    MODEL_DIRS=MODEL_DIRS,
    EXCEL_PATHS=EXCEL_PATHS,
)

In [ ]:
# ============================================================
# Cell 11: GCN 多GPU并行实验
# ============================================================
print("\n" + "=" * 50)
print("9.1 GCN 参数实验（多GPU并行）")
print("=" * 50)

gcn_combos = expand_param_grid(param_grids["GCN"])
gcn_results = run_parallel_experiments("GCN", gcn_combos, data, class_names)

if gcn_results:
    best_gcn = max(gcn_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGCN 最佳: hidden_dim={best_gcn['hidden_dim']}, dropout={best_gcn['dropout']}, "
        f"num_layers={best_gcn['num_layers']}, Test Acc={best_gcn['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Cell 12: GraphSAGE 多GPU并行实验
# ============================================================
print("\n" + "=" * 50)
print("9.2 GraphSAGE 参数实验（多GPU并行）")
print("=" * 50)

sage_combos = expand_param_grid(param_grids["GraphSAGE"])
sage_results = run_parallel_experiments("GraphSAGE", sage_combos, data, class_names)

if sage_results:
    best_sage = max(sage_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGraphSAGE 最佳: hidden_dim={best_sage['hidden_dim']}, dropout={best_sage['dropout']}, "
        f"num_layers={best_sage['num_layers']}, Test Acc={best_sage['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Cell 13: GATv2 多GPU并行实验
# ============================================================
print("\n" + "=" * 50)
print("9.3 GATv2 参数实验（多GPU并行）")
print("=" * 50)

gatv2_combos = expand_param_grid(param_grids["GATv2"])
gatv2_results = run_parallel_experiments("GATv2", gatv2_combos, data, class_names)

if gatv2_results:
    best_gatv2 = max(gatv2_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGATv2 最佳: hidden_dim={best_gatv2['hidden_dim']}, heads={best_gatv2['heads']}, "
        f"dropout={best_gatv2['dropout']}, num_layers={best_gatv2['num_layers']}, "
        f"Test Acc={best_gatv2['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Cell 14: GAT 多GPU并行实验
# ============================================================
print("\n" + "=" * 50)
print("9.4 GAT 参数实验（多GPU并行）")
print("=" * 50)

gat_combos = expand_param_grid(param_grids["GAT"])
gat_results = run_parallel_experiments("GAT", gat_combos, data, class_names)

if gat_results:
    best_gat = max(gat_results, key=lambda x: x["best_val_acc"])
    print(
        f"\nGAT 最佳: hidden_dim={best_gat['hidden_dim']}, heads={best_gat['heads']}, "
        f"dropout={best_gat['dropout']}, num_layers={best_gat['num_layers']}, "
        f"use_skip={best_gat['use_skip']}, Test Acc={best_gat['test_acc']:.4f}"
    )

In [ ]:
# ============================================================
# Cell 15: 结果汇总和可视化
# ============================================================
print("\n" + "=" * 60)
print("10. 结果汇总和可视化")
print("=" * 60)

all_best_models = {}
for model_name, results in [
    ("TopACT", topact_results),  # ★ TopACT 替换 MLP
    ("GCN", gcn_results),
    ("GraphSAGE", sage_results),
    ("GAT", gat_results),
    ("GATv2", gatv2_results),
]:
    if results:
        all_best_models[model_name] = max(results, key=lambda x: x["best_val_acc"])

if all_best_models:
    summary_data = []
    for model_name, best_model in all_best_models.items():
        summary_data.append(
            {
                "Model": model_name,
                "Best Val Acc": best_model["best_val_acc"],
                "Test Acc": best_model["test_acc"],
                "Test F1 (Macro)": best_model["test_f1_macro"],
                "Test F1 (Weighted)": best_model["test_f1_weighted"],
                "Precision (Macro)": best_model.get("test_precision_macro", 0),
                "Recall (Macro)": best_model.get("test_recall_macro", 0),
                "Hidden Dim": best_model["hidden_dim"],
                "Dropout": best_model["dropout"],
                "Layers": best_model["num_layers"],
                "Heads": best_model.get("heads", "N/A"),
                "Params": best_model.get("num_params", 0),
            }
        )

    summary_df = pd.DataFrame(summary_data).sort_values("Test Acc", ascending=False)
    print("\n模型性能对比表:")
    print(summary_df.to_string(index=False))

    summary_df.to_csv(os.path.join(RESULTS_DIR, "model_summary.csv"), index=False)
    print(f"\n✓ 汇总表已保存到 {RESULTS_DIR}")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    ax1 = axes[0, 0]
    bars = ax1.bar(summary_df["Model"], summary_df["Test Acc"], color="steelblue")
    ax1.set_ylabel("Accuracy")
    ax1.set_title("Model Test Accuracy Comparison")
    ax1.set_ylim(0, 1)
    for bar, acc in zip(bars, summary_df["Test Acc"]):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{acc:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    x = np.arange(len(summary_df))
    w = 0.35
    ax2.bar(
        x - w / 2,
        summary_df["Test F1 (Macro)"],
        w,
        label="Macro F1",
        color="forestgreen",
    )
    ax2.bar(
        x + w / 2,
        summary_df["Test F1 (Weighted)"],
        w,
        label="Weighted F1",
        color="lightcoral",
    )
    ax2.set_xticks(x)
    ax2.set_xticklabels(summary_df["Model"], rotation=45, ha="right")
    ax2.set_ylabel("F1 Score")
    ax2.set_title("Model F1 Score Comparison")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    ax3.scatter(summary_df["Params"] / 1000, summary_df["Test Acc"], s=100, alpha=0.6)
    for _, row in summary_df.iterrows():
        ax3.annotate(
            row["Model"],
            (row["Params"] / 1000, row["Test Acc"]),
            fontsize=8,
            ha="center",
        )
    ax3.set_xlabel("Number of Parameters (K)")
    ax3.set_ylabel("Test Accuracy")
    ax3.set_title("Model Efficiency: Parameters vs Accuracy")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    metrics = ["Test Acc", "Test F1 (Macro)", "Precision (Macro)", "Recall (Macro)"]
    num_metrics = len(metrics)
    angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
    angles += angles[:1]
    for _, row in summary_df.iterrows():
        values = [row[m] for m in metrics] + [row[metrics[0]]]
        ax4.plot(angles, values, "o-", linewidth=2, label=row["Model"], alpha=0.7)
    ax4.set_xticks(angles[:-1])
    ax4.set_xticklabels(metrics)
    ax4.set_ylim(0, 1)
    ax4.set_title("Multi-metric Radar Chart")
    ax4.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))
    ax4.grid(True)

    plt.suptitle(
        "Comprehensive Model Performance Comparison", fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.savefig(FIGURE_PATHS["test_comparison"], dpi=200, bbox_inches="tight")
    plt.show()

    best_model_name = summary_df.iloc[0]["Model"]
    print(f"\n🏆 最终最佳模型: {best_model_name}")
    print(f"   测试准确率: {summary_df.iloc[0]['Test Acc']:.4f}")
    print(f"   Macro F1: {summary_df.iloc[0]['Test F1 (Macro)']:.4f}")

print("\n" + "=" * 60)
print("实验完成！")
print(f"所有结果已保存到: {BASE_DIR}")
print(f"  - 模型文件: {MODELS_DIR}")
print(f"  - 训练曲线: {CURVES_DIR}")
print(f"  - 混淆矩阵: {CONFUSION_DIR}")
print(f"  - 结果汇总: {RESULTS_DIR}")
print("=" * 60)